## The Price Is Right - Building the Autonomous Planning Agent

In [1]:
# importing libraries
import json
from openai import OpenAI
from dotenv import load_dotenv
from agents.scanner_agent import ScannerAgent
import chromadb
import logging

In [2]:
# setting the environment

load_dotenv(override=True)
openai = OpenAI()
MODEL = 'gpt-5.1'

In [3]:
# starting with some test data
test_results = ScannerAgent().test_scan()
test_results

DealSelection(deals=[Deal(product_description="The Hisense R6 Series 55R6030N is a 55-inch 4K UHD Roku Smart TV that offers stunning picture quality with 3840x2160 resolution. It features Dolby Vision HDR and HDR10 compatibility, ensuring a vibrant and dynamic viewing experience. The TV runs on Roku's operating system, allowing easy access to streaming services and voice control compatibility with Google Assistant and Alexa. With three HDMI ports available, connecting multiple devices is simple and efficient.", price=178.0, url='https://www.dealnews.com/products/Hisense/Hisense-R6-Series-55-R6030-N-55-4-K-UHD-Roku-Smart-TV/484824.html?iref=rss-c142'), Deal(product_description='The Poly Studio P21 is a 21.5-inch LED personal meeting display designed specifically for remote work and video conferencing. With a native resolution of 1080p, it provides crystal-clear video quality, featuring a privacy shutter and stereo speakers. This display includes a 1080p webcam with manual pan, tilt, and

### Create 3 Pretend Functions for testing

In [5]:
def scan_the_internet_for_bargains() -> str:
    """This tool scans the internet for great deals and gets a curated list of promising deals"""

    print("Fake function to scan the internet - this returns a hardcoded set of deals")
    return test_results.model_dump_json()

In [6]:
def estimate_true_value(description: str) -> str:
    """This tool estimates the true value of a product based on a text description of it."""

    print(f"Fake function to estimating true value of {description[:20]}... - this always returns $300")
    return f"Product {description} has an estimated true value of 300"

In [7]:
def notify_user_of_deal(description: str, deal_price: float, estimated_true_value: float, url: str) -> str:
    """This tool notifies the user of a great deal, given a description of it, the price of the deal, and the estimated true value"""

    print(f"Fake function to notify user of {description} which costs {deal_price} and estimate is {estimated_true_value}")
    return "notification sent ok"

In [8]:
scan_the_internet_for_bargains()

Fake function to scan the internet - this returns a hardcoded set of deals


'{"deals":[{"product_description":"The Hisense R6 Series 55R6030N is a 55-inch 4K UHD Roku Smart TV that offers stunning picture quality with 3840x2160 resolution. It features Dolby Vision HDR and HDR10 compatibility, ensuring a vibrant and dynamic viewing experience. The TV runs on Roku\'s operating system, allowing easy access to streaming services and voice control compatibility with Google Assistant and Alexa. With three HDMI ports available, connecting multiple devices is simple and efficient.","price":178.0,"url":"https://www.dealnews.com/products/Hisense/Hisense-R6-Series-55-R6030-N-55-4-K-UHD-Roku-Smart-TV/484824.html?iref=rss-c142"},{"product_description":"The Poly Studio P21 is a 21.5-inch LED personal meeting display designed specifically for remote work and video conferencing. With a native resolution of 1080p, it provides crystal-clear video quality, featuring a privacy shutter and stereo speakers. This display includes a 1080p webcam with manual pan, tilt, and zoom contro

In [9]:
estimate_true_value("a new iPhone")

Fake function to estimating true value of a new iPhone... - this always returns $300


'Product a new iPhone has an estimated true value of 300'

In [10]:
notify_user_of_deal('a new iPhone', 100, 1000, 'https://www.apple.com/iphone')

Fake function to notify user of a new iPhone which costs 100 and estimate is 1000


'notification sent ok'

In [16]:
# Now, the JSON block
scan_function = {
    "name": "scan_the_internet_for_bargains",
    "description": "Returns top bargains from the internet along with the price each item is being offered for",
    "parameters": {
        "type": 'object',
        "properties": {},
        "required": [],
        "additionalProperties": False
    }
}

estimate_function = {
    "name": "estimate_true_value",
    "description": "Given the description of an item, estimate how much it is actually worth",
    "parameters": {
        "type": 'object',
        "properties": {
            'description': {
                'type': 'string',
                'description': 'The description of the item to be estimated'
            },
        },
        "required": ['description'],
        "additionalProperties": False
    }
}

notify_function = {
    "name": "notify_user_of_deal",
    "description": "Send the user a push notification about the single most compelling deal; only call this one time.",
    "parameters": {
        "type": 'object',
        "properties": {
            'description': {
                'type': 'string',
                'description': 'The description of the item itself scraped from the internet'
            },
            'deal_price': {
                'type': 'number',
                'description': 'The price offered by this deal scraped from the internet'
            },
            'estimated_true_value': {
                'type': 'number',
                'description': "The estimated actual value that this is worth"
            },
            'url': {
                'type': 'string',
                'description': 'The URL of this deal as scraped from the internet'
            }
        },
        "required": ['description', 'deal_price', 'estimated_true_value', 'url'],
        "additionalProperties": False
    }
}

In [17]:
tools = [
    {'type': 'function', 'function': scan_function},
    {'type': 'function', 'function': estimate_function},
    {'type': 'function', 'function': notify_function}
]

In [23]:
def handle_tool_call(message):
    """Actually call the tools associated with this message"""

    results = []
    for tool_call in message.tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        # execute the tool
        tool = globals().get(tool_name)

        if tool:
            tool_result = tool(**arguments)
        else:
            tool_result = {"error": f"Tool {tool_name} not found"}

        # Convert the result into text for the LLM
        if isinstance(tool_result, str):
            content = tool_result
        else:
            content = json.dumps(tool_result, default=str)

        results.append({"role": 'tool', "content": content, 'tool_call_id': tool_call.id})

    return results

In [19]:
system_message = "You find great deals on bargain products using your tools, and notify the user of the best bargain"

user_message = """
First, use your tool to scan the internet for bargain deals. Then, for each deal, use your tool to estimate its true value.
Then, pick the single most compelling deal where the price is much lower than the estimated true value, and use your tool to
notify the user.
Then, just reply OK to indicate success.
"""

messages = [
    {"role": 'system', 'content': system_message},
    {"role": 'user', "content": user_message}
]

In [24]:
done = False

while not done:
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=='tool_calls':
        message = response.choices[0].message
        results = handle_tool_call(message)
        messages.append(message)
        messages.extend(results)
    else:
        done = True

response.choices[0].message.content

Fake function to scan the internet - this returns a hardcoded set of deals
Fake function to estimating true value of The Hisense R6 Serie... - this always returns $300
Fake function to estimating true value of The Poly Studio P21 ... - this always returns $300
Fake function to estimating true value of The Lenovo IdeaPad S... - this always returns $300
Fake function to estimating true value of The Dell G15 gaming ... - this always returns $300
Fake function to notify user of The Poly Studio P21 is a 21.5-inch LED personal meeting display designed specifically for remote work and video conferencing. With a native resolution of 1080p, it provides crystal-clear video quality, featuring a privacy shutter and stereo speakers. This display includes a 1080p webcam with manual pan, tilt, and zoom control, along with an ambient light sensor to adjust the vanity lighting as needed. It also supports 5W wireless charging for mobile devices, making it an all-in-one solution for home offices. which c

'OK'

### The Autonomous Planning Agent

In [25]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [26]:
DB = 'products_vectorstore'
client = chromadb.PersistentClient(path=DB)
collection = client.get_or_create_collection('products')

In [27]:
from agents.autonomous_planning_agent import AutonomousPlanningAgent

agent = AutonomousPlanningAgent(collection)
agent.plan()

INFO:root:[Autonomous Planning Agent] Autonomous Planning Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Ensemble Agent] Initializing Ensemble Agent
INFO:root:[Specialist Agent] Specialist Agent is initializing - connecting to modal
INFO:root:[Frontier Agent] Initializing Frontier Agent
INFO:root:[Frontier Agent] Frontier Agent is setting up with OpenAI
INFO:sentence_transformers.base.model:No device provided, using mps
INFO:sentence_transformers.base.model:Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent is ready
INFO:root:[Neural Network Agent] Neural Network Agent is initializing
INFO:root:Neural Network is using mps
INFO:root:[Neural Network Agent] Neural Network Agent is ready and weights are loaded
INFO:root:[Ensemble Agent] Ensemble Agent is ready
INFO:root:[Messaging Agent] Messaging Agent is initializing
INFO:root:[Messaging Agent] Messaging Agent has initialized Pushover and Claude
INFO:root:[Autonomous Planning Agent] Autonomous Planning Agent is ready
INFO:root:[Autonomous Planning Agent] Autonomous Planning Agent is kicking off a run
INFO:root:[Autonomous Planning Agent] Autonomous Planning agent is calling scanner
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent received 30 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from 

20:33:50 - LiteLLM:INFO: utils.py:4502 - 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq


/Users/irenebusah/Desktop/ai-engineer-4-ds/lessons/llm-engineering/.venv/lib/python3.13/site-packages/pydantic/_internal/_generate_schema.py:1463: UserWarning: Item 'summary' on TypedDict class 'ChatCompletionReasoningItem' is using the `ReadOnly` qualifier. Pydantic will not protect items from any mutation on dictionary instances.
  warnings.warn(
INFO:LiteLLM:
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq


20:33:52 - LiteLLM:INFO: utils.py:1864 - Wrapper: Completed Call, calling success_handler


... (28 outputs left out)

INFO:LiteLLM:
LiteLLM completion() model= gpt-5-mini; provider = openai


20:35:07 - LiteLLM:INFO: utils.py:1864 - Wrapper: Completed Call, calling success_handler


INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification
INFO:root:[Messaging Agent] Messaging Agent has completed
INFO:root:[Autonomous Planning Agent] Autonomous Planning Agent completed with: OK


Opportunity(deal=Deal(product_description='Motorized outdoor roller shade with an integrated solar panel that powers a rechargeable motor. Available in widths from 2 ft to 16 ft, the shade uses blackout fabric that blocks 99% of UV rays, mounts with an aluminum alloy valance, and is controllable via a Bluetooth app or included 16-channel remote. It is compatible with Alexa and Google Assistant when paired with a Bluetooth hub and is rated for outdoor conditions with waterproof motor components.', price=115.0, url='https://www.dealnews.com/Motorized-Solar-Powered-Outdoor-Roller-Shade-for-115-free-shipping/22296433.html?iref=rss-f1912'), estimate=325.10216235351567, discount=210.10216235351567)